# Magenta RT — Holly Herndon (local GPU + API)

One notebook: **environment setup**, optional **JAX / Colab checks**, then a **local FastAPI** server for the p5 sketch. Ngrok is **not** started here; use **`gateway/gateway.ipynb`** so traffic hits `/magentart/...` on one public URL with MusicGen and Spleeter.

**Before you start**

- **Python 3.11+** kernel (see project `README` / `magentart-holly/README` for conda + `ipykernel`).
- Run Jupyter from the directory where **`magenta-realtime`** and **`t5x`** should be cloned (usually this repo or your project folder).
- Default API port **8103** (`MAGENTART_PORT` to override). Gateway env: `MAGENTART_UPSTREAM`.

**Flow:** run cells top to bottom once per fresh machine; re-run only the **server** cell after kernel restarts if deps are already installed.


## API packages (wheel-friendly)

Installs FastAPI and Uvicorn for the HTTP server. Heavy deps (`magenta-realtime`, JAX, t5x) come from the next section.


In [ ]:
%pip install -q fastapi "uvicorn[standard]" nest-asyncio


## 1. One-time setup (clones, patches, editable installs)

Clones **`magenta-realtime`** / **`t5x`** if missing, patches, installs GPU extras via **`sys.executable`**. Re-run after pulling new patch files from upstream.


In [ ]:
import subprocess
import sys
from pathlib import Path
import importlib.util as _iutil
ROOT = Path.cwd()
PY = sys.executable


def sh(cmd: str) -> None:
  subprocess.run(cmd, shell=True, cwd=ROOT, executable="/bin/bash", check=True)


def pip(*args: str) -> None:
  subprocess.run([PY, "-m", "pip", *args], cwd=ROOT, check=True)


if not (ROOT / "magenta-realtime").is_dir():
  sh("git clone https://github.com/magenta/magenta-realtime.git")
if not (ROOT / "t5x").is_dir():
  sh("git clone https://github.com/google-research/t5x.git")

subprocess.run(
    "patch -N t5x/setup.py < magenta-realtime/patch/t5x_setup.py.patch || true",
    shell=True,
    cwd=ROOT,
    executable="/bin/bash",
    check=True,
)
subprocess.run(
    "patch -N t5x/t5x/partitioning.py < magenta-realtime/patch/t5x_partitioning.py.patch || true",
    shell=True,
    cwd=ROOT,
    executable="/bin/bash",
    check=True,
)
for _rej in (ROOT / "t5x/setup.py.rej", ROOT / "t5x/t5x/partitioning.py.rej"):
  try:
    _rej.unlink()
  except OSError:
    pass

_setup = ROOT / "t5x/setup.py"
_t = _setup.read_text()
_t = _t.replace("_jax_version = '0.4.16'", "_jax_version = '0.8.1'")
_t = _t.replace("_jaxlib_version = '0.4.16'", "_jaxlib_version = '0.8.1'")
_setup.write_text(_t)

pip("install", "-U", "pip", "setuptools", "wheel")
pip("install", "chex>=0.1.85", "orbax-checkpoint>=0.6")
pip("install", "-e", "t5x[gpu]")
pip("install", "-e", "magenta-realtime[gpu]")
pip("install", "tf2jax==0.3.8")
pip("install", "ipywidgets>=8.0")
pip("install", "nest-asyncio")

_colab_shim = """try:
  colab = importlib.import_module("google.colab")
except ModuleNotFoundError:
  import types

  def _colab_register_callback(_name, _callback, **_kwargs):
    pass

  colab = types.SimpleNamespace(
      output=types.SimpleNamespace(register_callback=_colab_register_callback),
  )
"""

_mr = None
_spec_mr = _iutil.find_spec("magenta_rt")
if _spec_mr:
  if _spec_mr.submodule_search_locations:
    _mr = Path(list(_spec_mr.submodule_search_locations)[0]) / "colab"
  elif _spec_mr.origin:
    _mr = Path(_spec_mr.origin).resolve().parent / "colab"
if _mr is None and (ROOT / "magenta-realtime" / "magenta_rt" / "colab").is_dir():
  _mr = ROOT / "magenta-realtime" / "magenta_rt" / "colab"

if _mr and _mr.is_dir():
  _u = (_mr / "utils.py").read_text().replace("\r\n", "\n")
  _needle_u = "# using importlib to avoid build issues\ncolab = importlib.import_module(\"google.colab\")"
  if _needle_u in _u:
    (_mr / "utils.py").write_text(_u.replace(_needle_u, "# using importlib to avoid build issues\n" + _colab_shim))
  elif "except ModuleNotFoundError" not in _u:
    print("WARN: utils.py colab needle not found:", _mr / "utils.py")
  _w = (_mr / "widgets.py").read_text().replace("\r\n", "\n")
  _needle_w = "colab = importlib.import_module('google.colab')"
  if _needle_w in _w:
    (_mr / "widgets.py").write_text(_w.replace(_needle_w, _colab_shim))
  elif "except ModuleNotFoundError" not in _w:
    print("WARN: widgets.py colab needle not found:", _mr / "widgets.py")
  print("Colab shim patched under", _mr)
else:
  print("WARN: could not resolve magenta_rt/colab (run after pip install -e magenta-realtime)")


# t5x: allow checkpoint restore under Jupyter (nested asyncio loop)
_t5cp = ROOT / "t5x/t5x/checkpoints.py"
_t5ct = _t5cp.read_text()
if "jupyter-nested-asyncio" not in _t5ct:
  import re as _re_t5

  _t5pat = _re_t5.compile(r"(?m)^(\s*)leaves = asyncio\.run\(run\(\)\)\s*$")
  _t5m = _t5pat.search(_t5ct)
  if _t5m:
    _ind = _t5m.group(1)
    _t5repl = (
        _ind + "try:  # jupyter-nested-asyncio\n"
        + _ind + "  import nest_asyncio\n"
        + _ind + "  nest_asyncio.apply()\n"
        + _ind + "except Exception:\n"
        + _ind + "  pass\n"
        + _t5m.group(0)
        + "\n"
    )
    _t5cp.write_text(_t5pat.sub(_t5repl, _t5ct, count=1))
    print("Patched", _t5cp, "for nested asyncio (Jupyter)")
  else:
    print("WARN: could not find leaves = asyncio.run(run()) in", _t5cp)

_patch_paths = [ROOT / "jupyter_local_audio_utils_patch.py"]
_patch_paths += list(ROOT.glob("**/jupyter_local_audio_utils_patch.py"))
_patch_audio = next((p for p in _patch_paths if p.is_file()), None)
if _patch_audio is not None:
  import importlib.util as _iju_audio

  _spec_audio = _iju_audio.spec_from_file_location(
      "_magenta_jupyter_audio_patch", _patch_audio
  )
  _mod_audio = _iju_audio.module_from_spec(_spec_audio)
  _spec_audio.loader.exec_module(_mod_audio)
  _up_utils = None
  if _mr and (_mr / "utils.py").is_file():
    _up_utils = _mr / "utils.py"
  elif (ROOT / "magenta-realtime" / "magenta_rt" / "colab" / "utils.py").is_file():
    _up_utils = ROOT / "magenta-realtime" / "magenta_rt" / "colab" / "utils.py"
  if _up_utils:
    try:
      if _mod_audio.patch_magenta_colab_utils(_up_utils):
        print("Patched", _up_utils, "for Jupyter audio (ipywidgets.Output + inline init)")
    except RuntimeError as _e_audio:
      print("WARN: jupyter audio utils patch:", _e_audio)
  else:
    print("WARN: could not find magenta_rt/colab/utils.py for audio patch")
else:
  print("WARN: jupyter_local_audio_utils_patch.py not found under", ROOT)

print("Setup finished using", PY)


## 2. Seqio vocab + JAX devices

Skip importing `tensorflow_text`; confirm a GPU is visible.


In [ ]:
import os
from pathlib import Path

# Default: one visible GPU before `import jax` to avoid multi-GPU NCCL
# errors during t5x inference (e.g. ncclAllReduce invalid argument).
_single_visible_gpu = True  # @param {type:"boolean"}
if _single_visible_gpu and "CUDA_VISIBLE_DEVICES" not in os.environ:
  os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import nest_asyncio
nest_asyncio.apply()  # before JAX/t5x if Jupyter already has a loop

import seqio

vocab_path = Path(seqio.__file__).resolve().parent / "vocabularies.py"
lines = vocab_path.read_text().splitlines()
filtered = [ln for ln in lines if "import tensorflow_text as tf_text" not in ln]
if len(filtered) != len(lines):
  vocab_path.write_text("\n".join(filtered) + "\n")

import jax

print("JAX devices:", jax.devices())


## 3. Start API server

Loads the Holly checkpoint and listens on **8103**. Leave this cell running; start **`gateway/gateway.ipynb`** in another kernel for ngrok + `/magentart`.


In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import struct
import threading
import time
import uuid
from typing import Any

import numpy as np

if os.environ.get("MAGENTA_FORCE_SINGLE_GPU", "1") == "1" and "CUDA_VISIBLE_DEVICES" not in os.environ:
  os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import nest_asyncio

nest_asyncio.apply()


def _patch_seqio_vocabularies() -> None:
  try:
    import seqio
    from pathlib import Path

    vocab_path = Path(seqio.__file__).resolve().parent / "vocabularies.py"
    lines = vocab_path.read_text().splitlines()
    filtered = [ln for ln in lines if "import tensorflow_text as tf_text" not in ln]
    if len(filtered) != len(lines):
      vocab_path.write_text("\n".join(filtered) + "\n")
  except OSError:
    pass


_patch_seqio_vocabularies()

from fastapi import FastAPI, HTTPException, Request, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
import uvicorn

from magenta_rt import asset
from magenta_rt import audio as audio_lib
from magenta_rt import system

PCM_MAGIC = b"MRT\x01"
AUDIO_PROMPT_MAGIC = b"MRTA"
AUDIO_PROMPT_SR = 16000
MAX_AUDIO_PROMPT_SECONDS = 2.0


def _pcm_message(samples: np.ndarray, sample_rate: int) -> bytes:
  if samples.dtype != np.float32:
    samples = np.asarray(samples, dtype=np.float32)
  ch = samples.shape[1] if samples.ndim == 2 else 1
  if samples.ndim == 1:
    samples = samples[:, np.newaxis]
  n = samples.shape[0]
  flat = np.ascontiguousarray(samples.reshape(-1))
  hdr = struct.pack("<4sIIHH", PCM_MAGIC, int(sample_rate), int(n), int(ch), 0)
  return hdr + flat.astype(np.float32).tobytes()


def _parse_audio_prompt_body(raw: bytes) -> np.ndarray:
  if len(raw) >= 16 and raw[:4] == AUDIO_PROMPT_MAGIC:
    sr = int(struct.unpack("<I", raw[4:8])[0])
    frames = int(struct.unpack("<I", raw[8:12])[0])
    ch = int(struct.unpack("<H", raw[12:14])[0])
    if sr != AUDIO_PROMPT_SR:
      raise ValueError(f"sample_rate must be {AUDIO_PROMPT_SR}")
    if ch != 1:
      raise ValueError("audio prompt must be mono")
    need = 16 + frames * 4
    if len(raw) < need:
      raise ValueError("truncated audio prompt")
    return np.frombuffer(raw[16:need], dtype=np.float32).copy()

  if len(raw) % 4 != 0:
    raise ValueError("audio prompt bytes must be float32")
  return np.frombuffer(raw, dtype=np.float32).copy()


def _compute_style_embedding(
    mrt: system.MagentaRTBase,
    style_text: str,
    audio_prompt: np.ndarray | None,
    audio_prompt_weight: float,
    mean_weight: float,
    centroid_weights: list[float],
    mean_style: np.ndarray | None,
    centroids: np.ndarray | None,
) -> np.ndarray:
  text = (style_text or "").strip() or "a tree falls in the forest"
  emb = np.asarray(mrt.embed_style(text), dtype=np.float32)
  weighted = emb.copy()
  total_w = 1.0

  if audio_prompt is not None and audio_prompt_weight > 0:
    ap = np.asarray(audio_prompt, dtype=np.float32)
    audio_emb = np.asarray(mrt.embed_style(audio_lib.Waveform(ap, AUDIO_PROMPT_SR)), dtype=np.float32)
    weighted = weighted + audio_emb * float(audio_prompt_weight)
    total_w += float(audio_prompt_weight)

  if mean_style is not None and mean_weight > 0:
    weighted = weighted + mean_style * float(mean_weight)
    total_w += float(mean_weight)
  if centroids is not None:
    for i, w in enumerate(centroid_weights):
      if w > 0 and i < len(centroids):
        weighted = weighted + np.asarray(centroids[i], dtype=np.float32) * float(w)
        total_w += float(w)
  if total_w > 0:
    weighted = weighted / total_w
  return weighted


def _new_session_dict(centroid_count: int) -> dict[str, Any]:
  return {
      "mrt_state": None,
      "style_text": "a tree falls in the forest",
      "temperature": 1.1,
      "topk": 40,
      "guidance_weight": 5.0,
      "mean_weight": 1.0,
      "centroid_weights": [0.0] * int(centroid_count),
      "audio_prompt": None,
      "audio_prompt_weight": 0.0,
  }


def _apply_params_from_body(sess: dict[str, Any], body: dict[str, Any], centroid_count: int) -> None:
  if "style_text" in body:
    sess["style_text"] = str(body.get("style_text") or sess["style_text"])
  if "temperature" in body:
    sess["temperature"] = float(body["temperature"])
  if "topk" in body:
    sess["topk"] = int(body["topk"])
  if "guidance_weight" in body:
    sess["guidance_weight"] = float(body["guidance_weight"])
  if "mean_weight" in body:
    sess["mean_weight"] = float(body["mean_weight"])
  if "audio_prompt_weight" in body:
    sess["audio_prompt_weight"] = float(body["audio_prompt_weight"])
  cw = body.get("centroid_weights")
  if isinstance(cw, list):
    sess["centroid_weights"] = [
        float(cw[i]) if i < len(cw) else 0.0 for i in range(centroid_count)
    ]


def _sync_next_pcm(
    mrt: system.MagentaRTBase,
    sess: dict[str, Any],
    mean_style: np.ndarray | None,
    centroids: np.ndarray | None,
    centroid_count: int,
) -> bytes:
  style = _compute_style_embedding(
      mrt,
      sess["style_text"],
      sess.get("audio_prompt"),
      float(sess.get("audio_prompt_weight") or 0.0),
      sess["mean_weight"],
      sess["centroid_weights"],
      mean_style,
      centroids,
  )
  chunk, new_st = mrt.generate_chunk(
      state=sess["mrt_state"],
      style=style,
      seed=None,
      temperature=sess["temperature"],
      topk=sess["topk"],
      guidance_weight=sess["guidance_weight"],
  )
  sess["mrt_state"] = new_st
  samples = np.asarray(chunk.samples, dtype=np.float32)
  return _pcm_message(samples, int(chunk.sample_rate))


def _sync_snippet_pcm(
    mrt: system.MagentaRTBase,
    audio_prompt: np.ndarray,
    temperature: float,
    topk: int,
    guidance_weight: float,
) -> bytes:
  ap = np.asarray(audio_prompt, dtype=np.float32)
  style = np.asarray(mrt.embed_style(audio_lib.Waveform(ap, AUDIO_PROMPT_SR)), dtype=np.float32)
  chunk, _ = mrt.generate_chunk(
      state=None,
      style=style,
      seed=None,
      temperature=float(temperature),
      topk=int(topk),
      guidance_weight=float(guidance_weight),
  )
  samples = np.asarray(chunk.samples, dtype=np.float32)
  return _pcm_message(samples, int(chunk.sample_rate))


def _install_cors(app: FastAPI) -> None:
  raw = (os.environ.get("MAGENTA_CORS_ORIGINS") or "*").strip()
  if raw == "*":
    origins: list[str] = ["*"]
    allow_credentials = False
  else:
    origins = [o.strip() for o in raw.split(",") if o.strip()]
    if not origins:
      origins = ["*"]
    allow_credentials = os.environ.get("MAGENTA_CORS_ALLOW_CREDENTIALS", "0") == "1"
  expose = [
      h.strip()
      for h in os.environ.get(
          "MAGENTA_CORS_EXPOSE_HEADERS",
          "X-Magenta-Session,Content-Type,Content-Length",
      ).split(",")
      if h.strip()
  ]
  ah_raw = (os.environ.get("MAGENTA_CORS_ALLOW_HEADERS") or "*").strip()
  if ah_raw == "*":
    allow_headers: list[str] = ["*"]
  else:
    allow_headers = [h.strip() for h in ah_raw.split(",") if h.strip()]
    if not allow_headers:
      allow_headers = ["*"]
  app.add_middleware(
      CORSMiddleware,
      allow_origins=origins,
      allow_credentials=allow_credentials,
      allow_methods=["GET", "POST", "PUT", "PATCH", "DELETE", "OPTIONS", "HEAD"],
      allow_headers=allow_headers,
      expose_headers=expose,
      max_age=int(os.environ.get("MAGENTA_CORS_MAX_AGE", "600")),
  )


def build_app(
    mrt: system.MagentaRTBase,
    mean_style: np.ndarray | None,
    centroids: np.ndarray | None,
    centroid_count: int,
) -> FastAPI:
  app = FastAPI()
  _install_cors(app)
  infer_lock = asyncio.Lock()
  http_sessions: dict[str, dict[str, Any]] = {}
  http_sessions_lock = asyncio.Lock()

  @app.get("/")
  async def root() -> dict[str, Any]:
    return {
        "ok": True,
        "model": "magenta_rt holly",
        "centroid_count": centroid_count,
        "routes": [
            "GET /health",
            "GET /ngrok-free-warm",
            "POST /api/next-chunk",
            "POST /api/params",
            "POST /api/reset",
            "POST /api/audio-prompt",
            "POST /api/snippet-chunk",
            "WS /ws/stream",
        ],
    }

  @app.get("/ngrok-free-warm")
  async def ngrok_free_warm() -> Response:
    return Response(status_code=204, headers={"Cache-Control": "no-store"})

  @app.get("/health")
  async def health() -> dict[str, str]:
    return {"status": "ok", "model": "magenta_rt holly"}

  async def _resolve_http_session(body: dict[str, Any], request: Request) -> tuple[str, dict[str, Any]]:
    sid_in = body.get("session_id")
    if sid_in is None:
      sid_in = request.headers.get("X-Magenta-Session")
    async with http_sessions_lock:
      if sid_in is not None and str(sid_in).strip() != "":
        key = str(sid_in)
        if key not in http_sessions:
          raise HTTPException(status_code=404, detail="unknown session_id")
        return key, http_sessions[key]
      nid = str(uuid.uuid4())
      http_sessions[nid] = _new_session_dict(centroid_count)
      return nid, http_sessions[nid]

  @app.post("/api/params")
  async def api_params(request: Request) -> dict[str, Any]:
    body = await request.json()
    sid, sess = await _resolve_http_session(body, request)
    _apply_params_from_body(sess, body, centroid_count)
    return {"ok": True, "session_id": sid}

  @app.post("/api/reset")
  async def api_reset(request: Request) -> dict[str, Any]:
    body = await request.json()
    sid_in = body.get("session_id")
    if not sid_in:
      raise HTTPException(status_code=400, detail="session_id required")
    async with http_sessions_lock:
      if str(sid_in) not in http_sessions:
        raise HTTPException(status_code=404, detail="unknown session_id")
      http_sessions[str(sid_in)]["mrt_state"] = None
    return {"ok": True, "session_id": str(sid_in)}

  @app.post("/api/audio-prompt")
  async def api_audio_prompt(request: Request) -> Response:
    raw = await request.body()
    try:
      audio = _parse_audio_prompt_body(raw)
    except ValueError as e:
      raise HTTPException(status_code=400, detail=str(e))

    max_frames = int(AUDIO_PROMPT_SR * MAX_AUDIO_PROMPT_SECONDS)
    if audio.shape[0] > max_frames:
      audio = audio[-max_frames:]

    sid, sess = await _resolve_http_session({}, request)
    w = request.headers.get("X-Audio-Weight")
    if w is not None:
      try:
        sess["audio_prompt_weight"] = float(w)
      except ValueError:
        raise HTTPException(status_code=400, detail="X-Audio-Weight must be a float")

    sess["audio_prompt"] = np.asarray(audio, dtype=np.float32)
    return Response(
        status_code=204,
        headers={"X-Magenta-Session": sid, "Cache-Control": "no-store"},
    )

  @app.post("/api/snippet-chunk")
  async def api_snippet_chunk(request: Request) -> Response:
    raw = await request.body()
    try:
      audio = _parse_audio_prompt_body(raw)
    except ValueError as e:
      raise HTTPException(status_code=400, detail=str(e))

    t = request.headers.get("X-Temperature")
    k = request.headers.get("X-Topk")
    g = request.headers.get("X-Guidance")
    temperature = float(t) if t is not None else 1.3
    topk = int(k) if k is not None else 40
    guidance_weight = float(g) if g is not None else 5.0

    loop = asyncio.get_running_loop()

    async with infer_lock:

      def _run() -> bytes:
        return _sync_snippet_pcm(mrt, audio, temperature, topk, guidance_weight)

      pcm = await loop.run_in_executor(None, _run)

    return Response(
        content=pcm,
        media_type="application/octet-stream",
        headers={"Cache-Control": "no-store"},
    )

  @app.post("/api/next-chunk")
  async def api_next_chunk(request: Request) -> Response:
    body = await request.json()
    sid, sess = await _resolve_http_session(body, request)
    _apply_params_from_body(sess, body, centroid_count)
    loop = asyncio.get_running_loop()

    async with infer_lock:

      def _run() -> bytes:
        return _sync_next_pcm(mrt, sess, mean_style, centroids, centroid_count)

      pcm = await loop.run_in_executor(None, _run)
    return Response(
        content=pcm,
        media_type="application/octet-stream",
        headers={"X-Magenta-Session": sid, "Cache-Control": "no-store"},
    )

  @app.websocket("/ws/stream")
  async def ws_stream(websocket: WebSocket) -> None:
    await websocket.accept()
    sess = _new_session_dict(centroid_count)
    stop_event = asyncio.Event()
    loop = asyncio.get_running_loop()

    async def _receive_controls() -> None:
      while not stop_event.is_set():
        try:
          msg = await websocket.receive_text()
        except WebSocketDisconnect:
          stop_event.set()
          return
        try:
          body = json.loads(msg)
        except json.JSONDecodeError:
          await websocket.send_json({"type": "error", "detail": "control message must be JSON"})
          continue

        kind = body.get("type")
        if kind == "stop":
          stop_event.set()
          return
        if kind == "reset":
          sess["mrt_state"] = None
          continue

        params = body.get("params") if isinstance(body.get("params"), dict) else body
        _apply_params_from_body(sess, params, centroid_count)

    receiver = asyncio.create_task(_receive_controls())
    await websocket.send_json({"type": "ready", "centroid_count": centroid_count})

    try:
      while not stop_event.is_set():
        async with infer_lock:

          def _run() -> bytes:
            return _sync_next_pcm(mrt, sess, mean_style, centroids, centroid_count)

          pcm = await loop.run_in_executor(None, _run)
        await websocket.send_bytes(pcm)
        await asyncio.sleep(0)
    except WebSocketDisconnect:
      pass
    finally:
      stop_event.set()
      receiver.cancel()
      try:
        await receiver
      except asyncio.CancelledError:
        pass

  return app


import os
import threading
import time

import uvicorn

API_PORT = int(os.environ.get("MAGENTART_PORT", "8103"))
HOST = "0.0.0.0"

print("Fetching Holly checkpoint and style assets…")
checkpoint_dir = asset.fetch(
    "checkpoints/llm_large_holly_finetune.tar",
    is_dir=True,
    extract_archive=True,
    source="gcp",
)
mean_style = np.load(
    asset.fetch("finetune_features/holly_mean_style_embedding.npy", source="gcp")
)
cluster = np.load(
    asset.fetch("finetune_features/holly_cluster_centroids.npy", source="gcp")
)
centroid_count = int(cluster.shape[0])

print("Loading MagentaRT (lazy=False)…")
mrt = system.MagentaRT(
    tag="large",
    lazy=False,
    checkpoint_dir=checkpoint_dir,
)

app = build_app(mrt, mean_style, cluster, centroid_count)


def _run():
    uvicorn.run(app, host=HOST, port=API_PORT, log_level="info")


threading.Thread(target=_run, daemon=True).start()
time.sleep(2)
print(
    f"Magenta RT Holly API http://127.0.0.1:{API_PORT}/ "
    f"— route /magentart on the shared gateway (MAGENTART_PORT overrides port)."
)


## License

See the [Magenta RealTime repository](https://github.com/magenta/magenta-realtime) (Apache 2.0 code, CC-BY where noted).
